## Welcome to the UCLA V4 Miniscope Stripe Cleaner!

This jupyter notebook can be used to fix buffer mismatches and recover data you have collected. Stripes, as defined in this notebook, are any instance of vertical lines appearing in your Miniscope recording that should not be there. There are multiple types of stripes that can occur during Miniscope recordings, and this notebook takes different approaches to fix or remove the different types. Below is a brief description of the types of striping that can occur during recordings.

Stable block stripes occur when the packets of data become misaligned during acquisition. **Thankfully, these are fixable!** By re-arranging the the data correctly, you can recover your field of view and will not have to lose any data. This is the main goal of this notebook.

Moving stripes, which are often thin horizontal lines that move rapidly along the field of view, are likely caused by electrical noise. **These are not fixable.** In order to remove this noise from your recordings, you will either need to replace your coaxial cable, the button connecting to the PCB of the Miniscope, or the Miniscope PCB itself. There is a code block at the bottom of this notebook that you can use to replace those un-fixable frames with a previous frame without the stripes. Of course, if your whole recording has them, then it is impossible to remove them. Replacing with a previous good frame should only be done if it is a couple of frames of moving stripes (1-30 frames ideally for a 30 frames per second recording). Any larger replacement may cause problems in the preprocessing pipeline of your choice, as it will introduce a large chunk of completely stable fluorescence (since the frame without stripes is being copied multiple times).

To get started, first load the necessary packages below.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import shutil
import numpy as np
import holoviews as hv
import cleaning_functions as cl
from dask.distributed import (
    Client, 
    LocalCluster
)
from minian.io import load_videos
from minian.utilities import (
    save_minian,
    get_optimal_chk
)

hv.extension("bokeh", "matplotlib")

### 1. Set paths and parameters.

`dpath` is the path to your video that needs correction. `intpath` will create a temporary directory where an intermediate variable will be stored. It is removed at the end of the notebook.

`video_cropped` is an important parameter to keep in mind. The default sensor size for the V4 Miniscope is 608 x 608 pixels, but there are some configuration files floating around that have changed how many pixels you are recording from, such as 600 x 600 pixels. To check this, go to your Miniscope configuration json, scroll down to devices > miniscopes > roi, and check what your height and width are set to. Default would be 608 and 608. If it is anything different, you will need to update the `video_cropped` parameter from None to the appropriate height and width. To do so, you would create a dictionary as follows: `video_cropped = {'h' : (0, 600), 'w' : (0, 600)}`. If your configuration file is 608 x 608, set `video_cropped = None`.

`param_load_videos` is a dictionary that influences how your .avi files are loaded. You should **not** downsample temporally or spatially when stripe cleaning, so keep frame, height, and width set to 1.

In [ ]:
dpath = './demo_data'
intpath = './cleaner_intermediate'
video_cropped = None 

param_load_videos = {
    "pattern": r"[0-9]+\.avi$", 
    "dtype": np.uint8,
    "downsample": dict(frame=1, height=1, width=1),
    "downsample_strategy": "subset",
}

### 2. Start a local cluster on your machine to manage memory usage.

In [ ]:
cluster = LocalCluster(n_workers=4, 
                       memory_limit="6GB",
                       threads_per_worker=2)
client = Client(cluster)
client

### 3. Load and visualize your session to check for stripes.

We recommend visualizing the maximum projection of your video after you've ran it through the stripe cleaner to ensure that the stripes were actually corrected.

In [ ]:
varr = load_videos(dpath, **param_load_videos)
chk, _ = get_optimal_chk(varr, dtype=float)
varr = save_minian(
    varr.chunk({"frame": chk["frame"], "height": -1, "width": -1}).rename("varr"),
    intpath,
    overwrite=True,
)
varr_ref = varr

In [ ]:
## Plot max projection to observe absence or presence of stripes
im_opts = dict(
    frame_width = 500,
    aspect = varr_ref.sizes['width'] / varr_ref.sizes['height'],
    cmap = 'Viridis',
    colorbar = True,
)
(
    hv.Image(
        varr_ref.max('frame').compute().astype(float),
        ['width', 'height'],
        label = 'Maximum Projection',
    ).opts(**im_opts)
)

### 4. Detect stripes.

If you find that you aren't detecting all the stripes in your video, you will have to change the `thresh` argument in the function `no_stripes_frames`. Generally, 13-18 works well for detecting stripes. If there is a single horizontal stripe in your video, it won't be detected using this method. See the optional code blocks below.

In [ ]:
## Find frames with misaligned rows
frames_without_stripes = cl.no_stripes_frames(varr_ref, thresh=16).values
bad_frames = np.asarray(varr_ref[~frames_without_stripes].frame)
fnames, frame_numbers = cl.get_filename(dpath, bad_frames)

print(f'Bad frames: {bad_frames}')
print(f'Number of bad frames: {len(bad_frames)}')
print(f'Files with stripes: {fnames}')

In [ ]:
## OPTIONAL CODE BLOCK ##

## Use this block to visualize single frames in a video if you are unable to detect singular stripes in your videos 
## Change frame range (frame_start, frame_end). 
## Frame range is relative to all the frames in your full recording (for example, frame 22000 to 22130)

frame_start = 390
frame_end = 420
frame_range = np.arange(frame_start, frame_end, 1)
frame_dict = {f:cl.vid_frame(f, varr_ref) for f in frame_range}

im_opts = dict(frame_width=500, aspect=1, cmap='viridis')
hmap = hv.HoloMap(frame_dict, kdims='frame')
hmap.opts(**im_opts)

In [ ]:
## OPTIONAL CODE BLOCK ##

## If you use this code block, you MUST SKIP DIRECTLY TO STEP 6
## Use this block to manually choose which frames to correct in a video based on the visualization above
## The frame numbers must be relative to the number of frames in one .avi file (0 - 1000, since there's 1000 frames per .avi)
# frame_numbers = [np.array([190])] ## if single frame, must be in this style
# frame_numbers = [np.array(np.arange(715, 730))] ## if doing multiple frames, must be in this style

fnames = [os.path.join(dpath, '0.avi')] ## choose the .avi file that needs frames replaced
frame_numbers [np.array(np.arange(715, 730))] ## choose the frames that will be replaced, relative from 0 to 1000
print(fnames)
print(frame_numbers)

### 5. Realign data to remove stripes.

The function `fix_video` creates a new folder specified by the argument `folder_name` (by default called originals) and moves the original data files to said folder. This will ensure that you don't lose the original recordings and can revert back to them if necessary. The function will then write a new video file with the same name to replace the file that was moved, but the frames will be corrected in the new video.

The `offset` argument is the most important argument in this function. The block stripes are caused either by a **negative or positive** misalignment of the data. You will have to test by adding a negative sign (-) or not to the buffer size of 8184, running the function, and seeing if it corrected the stripes. If it did not, you would use the opposite of the negative or positive value you used before.

**Note**: Make sure the `compressionCodec` argument matches the compression code used to acquire your videos. You can find this in the Miniscope config file used for your mouse.

**Note**: We have observed one instance of a recording that was shifted by a factor of `offset = -(2 * 8184)`, so it is worth testing the double shifted version of the negative or positive offset. It would be very rare.

**Important user note**: Many (but not all) positive shifts are often preceeded by transient moving stripes that then jump to block stripes. As noted at the beginning of the notebook, moving stripes cannot be fixed and must be replaced or dropped. In these instances, the positive shifted frames will be fixed, but you will have to run the recording through this notebook for a second time to replace the transient moving stripes with a good frame (step 6 - replace bad frames).

In [ ]:
## Fix video
## Offset will be iterations of the buffer size, which is 8184
offset = 8184
cl.fix_video(fnames, 
             frame_numbers, 
             video_cropped=video_cropped, 
             folder_name='originals',
             offset=offset,
             compressionCodec='FFV1')

At this point, inspect the new videos and compare them to the ones in the "originals" folder. If the new ones look pristine, you're done! If not, follow these steps:
1. Cut or copy the files from the originals folder back to your dpath.
2. Replace the newly created files with the original .avi files.
3. Test a new offset. For example, if you tried 8184 and that did not work, now try -8184. 
4. If still not corrected, repeat steps 1-3 with double shifts.
5. If no values are fixing the block stripes, you have found a new type! You can try switching the argument `buffertofix` to 2 and trying different offsets, or try a positive/negative triple shift. 
6. If the **stationary** block stripes are not fixed with any of the above steps, make sure to check that your `video_cropped` argument is correct. 
7. Ultimately, if nothing is fixing them, you will have to use the optional step below to replace them with a good frame.

### 6. (Optional) Replace bad frames.
Running the cell below will create another directory "failed_to_fix." It will move the files you recently created above that contain stripes, and it will move those to the new "failed_to_fix" folder. Then it will write new videos that will replace the striped frames with the last good frame that occurs the frame before the bad frames. Make sure the `compressionCodec` argument matches the one you used for your data acquisition. After this finishes, check the new video files to see if they look okay. 

After using this notebook, we recommend removing your intpath below, closing your cluster, and restarting this notebook to plot the maximum intensity projection of your recording to visually inspect if all the stripes have been removed.

In [ ]:
## Write new videos by replacing the bad frames with the last good one
cl.rewrite_video(fnames, 
                 frame_numbers,
                 folder_name='failed_to_fix',
                 compressionCodec='FFV1')

### 7. Remove intpath and close cluster.

In [ ]:
## Remove intpath
shutil.rmtree(intpath)

In [ ]:
## Close cluster
client.close()
cluster.close()